# Week 3 Project: Implementing and Comparing Classification Techniques
**Project context:** EcoSort – AI-Based Waste Classification  
**Course:** DCS 410: Basics of Neural Networks and Deep Learning  
**Team:** Satkar Khadka, Sujwal Khadka, Suvekshya Jyu Thakuri  
**Institution:** Westcliff University

This Google Colab notebook implements **Gaussian Naive Bayes (generative)**, **Logistic Regression**, and **RBF Support Vector Machine (discriminative)** on **Digits** and **Iris** datasets. All three classifiers are evaluated on **both** datasets for a fair comparison. The datasets are built into scikit-learn, so **no uploads or downloads are required**.

**Note:** This is a classification-methods experiment supporting EcoSort, **not** a trained waste-image model. Results should not be represented as EcoSort waste classification performance.

## 1. Setup
Google Colab normally includes these packages. Run this cell if any dependency is missing.

In [ ]:
%pip -q install numpy pandas matplotlib scikit-learn

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.datasets import load_digits, load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, ConfusionMatrixDisplay, log_loss)

SEED = 42
OUT = Path('/content/week3_results') if Path('/content').exists() else Path('week3_results')
OUT.mkdir(parents=True, exist_ok=True)
print('Results directory:', OUT.resolve())

## 2. Datasets and an 80/20 stratified split
- **Digits**: 1,797 examples of 8×8 grayscale handwritten digits, 10 classes.
- **Iris**: 150 flower observations, four numeric features, three classes.

The same fixed train/test split is used for each classifier within a dataset. The test set is not used in training. Standardization is fitted **only on the training data** inside discriminative-model pipelines.

In [ ]:
datasets = {}
for name, loader in [('Digits', load_digits), ('Iris', load_iris)]:
    bunch = loader()
    X = np.asarray(bunch.data, dtype=float)
    y = np.asarray(bunch.target)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=SEED, stratify=y
    )
    datasets[name] = {'bunch': bunch, 'X_train': X_train, 'X_test': X_test,
                      'y_train': y_train, 'y_test': y_test}
    print(f'{name}: {len(y)} examples; {X.shape[1]} features; '
          f'{len(np.unique(y))} classes; train={len(y_train)}, test={len(y_test)}')

## 3. Define classifiers
**Generative:** Gaussian Naive Bayes models the class-conditional feature distributions and class priors. **Discriminative:** Logistic Regression directly models class probabilities, while the RBF SVM learns separating boundaries in a kernel feature space. The SVM probability estimates are fitted additionally for log-loss/risk analysis.

In [ ]:
def classifiers():
    return {
        'Gaussian Naive Bayes': GaussianNB(var_smoothing=1e-9),
        'Logistic Regression': make_pipeline(
            StandardScaler(), LogisticRegression(max_iter=3000, random_state=SEED)
        ),
        'RBF SVM': make_pipeline(
            StandardScaler(), SVC(kernel='rbf', C=1.0, gamma='scale',
                                  probability=True, random_state=SEED)
        ),
    }

models = {}
records = []
confusion_matrices = {}

## 4. Train and evaluate all models on both datasets
Report accuracy, macro-averaged precision/recall/F1, and multiclass log loss. Save confusion matrices. For Iris, compare standard predictions with a **demonstration** of cost-sensitive decisions.

In [ ]:
for dataset_name, data in datasets.items():
    Xtr, Xte = data['X_train'], data['X_test']
    ytr, yte = data['y_train'], data['y_test']
    models[dataset_name] = {}
    for model_name, model in classifiers().items():
        model.fit(Xtr, ytr)
        pred = model.predict(Xte)
        proba = model.predict_proba(Xte)
        prec, rec, f1, _ = precision_recall_fscore_support(
            yte, pred, average='macro', zero_division=0
        )
        row = {
            'Dataset': dataset_name, 'Model': model_name,
            'Train N': len(ytr), 'Test N': len(yte),
            'Accuracy': accuracy_score(yte, pred),
            'Macro Precision': prec, 'Macro Recall': rec, 'Macro F1': f1,
            'Log Loss': log_loss(yte, proba, labels=model.classes_)
        }
        if dataset_name == 'Iris':
            # Cost matrix L[true_class, predicted_class]:
            # Misclassifying a true virginica costs 3, other errors cost 1.
            L = np.ones((3, 3))
            np.fill_diagonal(L, 0)
            L[2, :] = 3
            L[2, 2] = 0
            # Expected loss of each candidate action a: sum_y P(y|x)*L[y,a]
            risks = proba @ L
            decisions = np.argmin(risks, axis=1)
            row['Default Mean Cost'] = float(np.mean(L[yte, pred]))
            row['Cost-Sensitive Mean Cost'] = float(np.mean(L[yte, decisions]))
            row['Cost-Sensitive Accuracy'] = accuracy_score(yte, decisions)
        records.append(row)
        models[dataset_name][model_name] = model
        confusion_matrices[(dataset_name, model_name)] = confusion_matrix(
            yte, pred, labels=model.classes_
        )

metrics = pd.DataFrame.from_records(records)
display(metrics.round(4))
metrics.to_csv(OUT / 'model_metrics.csv', index=False)

## 5. Compare performance visually

In [ ]:
for dataset_name in datasets:
    part = metrics[metrics['Dataset'] == dataset_name]
    fig, ax = plt.subplots(figsize=(8, 4))
    x = np.arange(len(part))
    width = 0.24
    for i, metric in enumerate(['Accuracy', 'Macro Precision', 'Macro F1']):
        ax.bar(x + (i-1)*width, part[metric], width=width, label=metric)
    ax.set_xticks(x, ['Naive Bayes', 'Logistic Reg.', 'RBF SVM'])
    ax.set_ylim(0, 1.06)
    ax.set_ylabel('Held-out test score')
    ax.set_title(f'{dataset_name}: classifier comparison')
    ax.legend(loc='lower right')
    fig.tight_layout()
    fig.savefig(OUT / f'comparison_{dataset_name.lower()}.png', dpi=180)
    plt.show()

## 6. Confusion matrices

In [ ]:
for dataset_name in datasets:
    for model_name in classifiers():
        model = models[dataset_name][model_name]
        fig, ax = plt.subplots(figsize=(5.3, 4.6))
        ConfusionMatrixDisplay(
            confusion_matrices[(dataset_name, model_name)],
            display_labels=model.classes_
        ).plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
        ax.set_title(f'{dataset_name}: {model_name}')
        fig.tight_layout()
        fig.savefig(OUT / f'cm_{dataset_name.lower()}_{model_name.lower().replace(" ", "_")}.png', dpi=160)
        plt.show()

## 7. What the generative model learns
Naive Bayes estimates each class prior, mean and variance for every feature, assuming conditionally independent Gaussian features: $P(x\mid y)=\prod_j P(x_j\mid y)$. Bayes' rule combines these terms into posterior class probabilities. The class-mean pixel patterns below illustrate what the model learned on Digits.

In [ ]:
nb_digits = models['Digits']['Gaussian Naive Bayes']
fig, axes = plt.subplots(2, 5, figsize=(10.5, 4.6))
for c, ax in enumerate(axes.flat):
    ax.imshow(nb_digits.theta_[c].reshape(8, 8), cmap='viridis', vmin=0, vmax=16)
    ax.set_title(f'Digit {c}')
    ax.axis('off')
fig.suptitle('Gaussian NB: estimated mean pixel intensity by digit class')
fig.tight_layout()
fig.savefig(OUT / 'nb_digit_class_means.png', dpi=170)
plt.show()

iris = datasets['Iris']['bunch']
nb_iris = models['Iris']['Gaussian Naive Bayes']
means = pd.DataFrame(nb_iris.theta_, columns=iris.feature_names, index=iris.target_names)
print('Iris class-conditional feature means:')
display(means.round(3))
means.to_csv(OUT / 'iris_nb_class_means.csv')
print('Estimated Iris class priors:', nb_iris.class_prior_.round(3))

## 8. Decision theory: minimizing expected loss
Under zero-one loss, predicting the most probable class minimizes conditional expected misclassification risk. For an alternative loss matrix $L(y,a)$, choose the action with the smallest $\sum_y P(y\mid x)L(y,a)$. The Iris example deliberately assigns a **threefold penalty** to mistakes on true *virginica* (all other mistakes cost 1). This illustrates how prediction decisions can change when costs matter; the matrix is hypothetical and not a real recycling policy.

In [ ]:
display(metrics.loc[metrics.Dataset == 'Iris',
                    ['Model', 'Accuracy', 'Default Mean Cost',
                     'Cost-Sensitive Mean Cost', 'Cost-Sensitive Accuracy']].round(4))
print('Caution: probabilities, especially Naive Bayes and SVM estimates, may be miscalibrated.')

## 9. Findings and relevance to EcoSort
1. Compare **all three classifiers on both datasets**, rather than attributing performance to the classifier family based on different datasets alone.
2. Gaussian Naive Bayes is fast and interpretable, but correlated pixel features challenge its independence assumption.
3. Logistic Regression and SVM learn discriminative relationships and can outperform the Gaussian model on handwritten-digit patterns.
4. Decision thresholds depend on the loss assigned to different errors; accuracy alone does not describe every deployment objective.
5. **EcoSort next step:** apply these baselines to a consistent representation of actual labeled waste images, then compare them with the proposed CNN/MobileNetV2. The Digits/Iris experiment does not assess waste-image accuracy.

**Reproducibility:** split seed 42, stratified 80/20 split, built-in scikit-learn datasets, default GaussianNB smoothing, standardized discriminative pipelines, RBF SVM with `C=1.0`. No cross-validation or hyperparameter tuning was performed. Small test sets and single splits are limitations.

## 10. Download output files (optional)
All metrics, learned class means, and plots are in the `week3_results` folder. In Google Colab, run this optional cell to package and download them.

In [ ]:
# Uncomment to download results as a ZIP directly from Colab:
# import shutil
# from google.colab import files
# zip_path = shutil.make_archive('/content/week3_results', 'zip', root_dir=OUT)
# files.download(zip_path)
print('Generated files:')
for output_file in sorted(OUT.iterdir()):
    print(' •', output_file.name)